<a href="https://colab.research.google.com/github/eshernan/big-data-postgraduate/blob/dev/supersalud/ProfesorSesion4Supersalud_BigData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Cuidado, esta es la copia del Profesor

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

# Instalación de librerías

In [ ]:
!pip -q install polars pyarrow psutil streamz dask[dataframe]

In [ ]:
import pyarrow as pa
import pyarrow.csv as pv
import pyarrow.parquet as pq

## Crear el archivo de parametrización

In [ ]:
# Definir el diccionario de configuración
config = {
    "urls": {
        "2023S2": "https://mapas.supersalud.gov.co/arcgisportal/sharing/rest/content/items/515d4c6366854f549b24c704320f10f4/data",
        "2024S1": "https://mapas.supersalud.gov.co/arcgisportal/sharing/rest/content/items/b52330a16b3940c39d38cb164c9dc014/data",
        "2024S2": "https://mapas.supersalud.gov.co/arcgisportal/sharing/rest/content/items/b257c9907b754b0cabb55a32e706bdee/data"

    },
    "demografia": {
        "P2025": "https://terridata.dnp.gov.co/assets/docs/excel/dimensiones/TerriData_Dim5.xlsx.zip"
    },
    "notify": {
        "email_to": "alertas-datos@supersalud.gov.co",
        "email_from": "pipelines@supersalud.gov.co",
        "smtp_host": "smtp.gmail.com",
        "smtp_port": 587,
        "use_tls": True
    },
    "storage": {
        "base_dir": "parquet_output",
        "format": "parquet",
        "compression": "snappy",
        "partitions": ["periodo", "mes"]
    }
}

# Guardar como archivo JSON
with open("config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=4, ensure_ascii=False)

print("Archivo config.json creado correctamente")

Archivo config.json creado correctamente


## Uso de Polars, para mejorar desempeño

In [ ]:
import polars as pl
import requests, io, json

## Validaciones de las URL y los separadores de campo en los csv

- Se verifica que se puedan leer las URL
- Se verifica que se pueda leer el CSV con un separador de campo, dentro de la lista
- Se verifica que la URL leida se pueda leer con el separador de campos definido

In [ ]:
# Función (tarea), para leer los archivos de datos, que fueron colocados en el json de configuración

def read_csv_auto(bytes_content: bytes) -> pl.DataFrame:
  print("Iniciando")
  attempts = []
  for sep in [";", ",","|"]:
      try:
          df = pl.read_csv(io.BytesIO(bytes_content), separator=sep, infer_schema_length=10000)
          attempts.append((sep, df))
      except Exception:
          continue
  if not attempts:
      raise ValueError("No se pudo leer el archivo con ; ni , ni |")
  # heurística simple: más columnas ⇒ mejor
  best = max(attempts, key=lambda x: len(x[1].columns))
  print(f"Archivo leído con el separador {best[0]}")
  return best[1]

def load_from_url(url: str) -> pl.DataFrame:
  r = requests.get(url, timeout=60, allow_redirects=True, verify=False)
  r.raise_for_status()
  return read_csv_auto(r.content)


## Validaciones y establecimiento de esquemas

In [ ]:
def assert_same_schema(dfs: list[pl.DataFrame]) -> list[str]:
    """
    Normaliza todos los nombres de columnas a minúsculas
    y valida que todos los DataFrames tengan el mismo esquema.

    Retorna la lista de columnas base si todo es consistente.
    """
    # Paso 1: normalizar nombres a minúscula en todos los DataFrames
    dfs_normalized = [df.rename({c: c.lower() for c in df.columns}) for df in dfs]

    # Paso 2: validar esquemas
    colsets = [set(df.columns) for df in dfs_normalized]
    base = colsets[0]
    if not all(cols == base for cols in colsets[1:]):
        diffs = []
        for i, cols in enumerate(colsets):
            missing = base - cols
            extra = cols - base
            if missing:
                diffs.append(f"DF[{i}] le faltan columnas: {missing}")
            if extra:
                diffs.append(f"DF[{i}] tiene columnas extra: {extra}")
        raise ValueError("Esquema distinto entre archivos:\n" + "\n".join(diffs))

    return list(dfs_normalized[0].columns)



## Definición de Typos de datos

In [ ]:
def coerce_types(df: pl.DataFrame) -> pd.DataFrame:
    col_int_nullable = [
        "pet_cod_depto", "pet_cod_mpio", "afec_cod_depto", "afec_cod_mpio",
        "ent_cod_depto", "ent_cod_mpio", "ent_cod_sns",
        "cod_macromot", "cod_motgen", "cod_motesp", "cod_tipo_motesp", "cod_subtipo_motesp",
        "mes", "afec_edad"
    ]
    for c in col_int_nullable:
        if c in df.columns:
            df = df.with_columns(pl.col(c).cast(pl.Int64, strict=False))
    if "periodo" in df.columns:
        df = df.with_columns(pl.col("periodo").cast(pl.Utf8, strict=False))
    # Normalización de género
    if "afec_genero" in df.columns:
        mapping = { "M":"Hombre","H":"Hombre","Masculino":"Hombre","Hombre":"Hombre",
                    "F":"Mujer","Femenino":"Mujer","Mujer":"Mujer" }
        df = df.with_columns(
            pl.col("afec_genero").cast(pl.Utf8, strict=False).str.strip().map_dict(mapping, default=None).alias("afec_genero")
        )
    return df


## Escritura en Parquet particionado por periodo y mes

Parquet comprime, acelera lecturas y estandariza tipos.

Particionar por **periodo** y **mes** permite cortes temporales rápidos (ej., vigilancia mensual, comparativos semestrales).

## Esquema de particionamiento sin formato Hive, que no incluye el nombre del campo en el folder

In [ ]:
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import os

def write_partitioned_parquet(df, base_dir, partitions, compression="snappy"):
    """
    Escribe un DataFrame de Polars a Parquet particionado usando PyArrow Dataset API.
    """
    os.makedirs(base_dir, exist_ok=True)

    # convertir Polars a Arrow
    table = df.to_arrow()

    # Configurar opciones de escritura de Parquet
    pq_options = pq.ParquetWriter  # solo para dejar claro, no es necesario instanciar aquí

    ds.write_dataset(
        data=table,
        base_dir=base_dir,
        partitioning=partitions,
        format="parquet",
        existing_data_behavior="overwrite_or_ignore"
    )


## Particionamiento con esquema de Hive, para incluir el campo en el nombre de la partición

In [ ]:
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import os

def write_partitioned_parquet(df, base_dir, partitions, compression="snappy"):
    """
    Escribe un DataFrame de Polars a Parquet particionado usando PyArrow Dataset API.
    """
    os.makedirs(base_dir, exist_ok=True)

    # convertir Polars a Arrow
    table = df.to_arrow()

    # 2) Construir schema de particiones y activar flavor "hive"
    part_fields = [table.schema.field(name) for name in partitions]
    part_schema = pa.schema(part_fields)
    hive_part = ds.partitioning(part_schema, flavor="hive")

    ds.write_dataset(
        data=table,
        base_dir=base_dir,
        partitioning=hive_part,
        format="parquet",
        existing_data_behavior="overwrite_or_ignore"
    )



## Generación de Reglas
### Reglas de terminación anticipada

Si **pet_cod_depto** o **pet_cod_mpio tienen** > 30% nulos (umbral en config.json) ⇒ cancelar pipeline y enviar correo.

**Justificación:** la capacidad de territorializar reclamos es clave para vigilancia diferencial y alertas regionales; sin esa georreferenciación mínima, la analítica posterior pierde validez.

# Generación del pipeline


In [ ]:
import os, json

# 1) Cargar config
with open("config.json","r") as f:
    CFG = json.load(f)

urls = CFG["urls"]
noti = CFG["notify"]
store= CFG["storage"]
demografia = CFG["demografia"]


In [ ]:
for name, url in urls.items():
  print("url {1} en archivo".format(name, url))
for name, url in demografia.items():
  print("url {1} en archivo".format(name, url))


url https://mapas.supersalud.gov.co/arcgisportal/sharing/rest/content/items/515d4c6366854f549b24c704320f10f4/data en archivo
url https://mapas.supersalud.gov.co/arcgisportal/sharing/rest/content/items/b52330a16b3940c39d38cb164c9dc014/data en archivo
url https://mapas.supersalud.gov.co/arcgisportal/sharing/rest/content/items/b257c9907b754b0cabb55a32e706bdee/data en archivo
url https://terridata.dnp.gov.co/assets/docs/excel/dimensiones/TerriData_Dim5.xlsx.zip en archivo


In [ ]:

# 2) Descargar / leer datasets de casos SuperSalud
dfs = []
for name, url in urls.items():
    df = load_from_url(url)
    print("Descargando archivo {1}".format(name, url))
    # aseguramos columnas periodo/mes presentes; si no, inferir de archivo o agregar
    if "periodo" not in df.columns:
        df = df.with_columns(pl.lit(name).alias("periodo"))  # p.ej., "2024S2"
    if "mes" not in df.columns:
        # si el archivo no trae "mes", pero el dataset es semestral, podríamos derivar de una columna de fecha si existe.
        # En demo, dejamos null y permitimos escribir; ideal: inferir.
        df = df.with_columns(pl.lit(None).cast(pl.Int64).alias("mes"))
    #df = coerce_types(df)
    dfs.append(df)

Iniciando
Archivo leído con el separador ;
Descargando archivo https://mapas.supersalud.gov.co/arcgisportal/sharing/rest/content/items/515d4c6366854f549b24c704320f10f4/data
Iniciando
Archivo leído con el separador ,
Descargando archivo https://mapas.supersalud.gov.co/arcgisportal/sharing/rest/content/items/b52330a16b3940c39d38cb164c9dc014/data
Iniciando
Archivo leído con el separador ;
Descargando archivo https://mapas.supersalud.gov.co/arcgisportal/sharing/rest/content/items/b257c9907b754b0cabb55a32e706bdee/data


### Datos de poblacion por DANE

In [ ]:
def cargar_poblacion_2018():
    # URL del archivo
    url = "https://www.dane.gov.co/files/censo2018/informacion-tecnica/CNPV-2018-VIHOPE-v2.xls"

    # Leemos el Excel desde la URL
    df = pd.read_excel(
        url,
        sheet_name="Población CNPV 2018",
        header=8  # fila 9 (Python indexa desde 0)
    )

    # Guardamos en el DataFrame solicitado
    poblacion_2018 = df.copy()

    return poblacion_2018


In [ ]:
df_demografia = cargar_poblacion_2018()

In [ ]:
df_demografia.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1159 entries, 0 to 1158
Data columns (total 6 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Código DIVIPOLA                     1157 non-null   object 
 1   NOMBRE DEPARTAMENTO                 1156 non-null   object 
 2   NOMBRE MUNICIPIO                    1122 non-null   object 
 3   Población en hogares particulares   1156 non-null   float64
 4   Población en LEA                    1156 non-null   float64
 5   Población total                     1156 non-null   float64
dtypes: float64(3), object(3)
memory usage: 54.5+ KB


In [ ]:
poblacion_quality_df = df_demografia

dtype_mapping = {
    'Código DIVIPOLA': 'Int32',
    'Población en hogares particulares ': 'Int32',
    'Población en LEA': 'Int32',
    'Población total': 'Int32',
}

for col, dtype in dtype_mapping.items():
    if dtype.startswith('Int'):
        poblacion_quality_df[col] = pd.to_numeric(poblacion_quality_df[col], errors='coerce').astype(dtype)
    else:
        poblacion_quality_df[col] = poblacion_quality_df[col].astype(dtype)

# Convertir los objetos en Categorias
categorical_cols = [
    'NOMBRE DEPARTAMENTO', 'NOMBRE MUNICIPIO'
]

for col in categorical_cols:
    # Vallidar si la columna existe antes de convertir
    if col in poblacion_quality_df.columns:
        poblacion_quality_df[col] = poblacion_quality_df[col].astype('category')

# Renombrar Columna
poblacion_quality_df.rename(columns={'Código DIVIPOLA': 'cod_divipola'}, inplace=True)
poblacion_quality_df.rename(columns={'Población en hogares particulares ': 'poblacion_hogares'}, inplace=True)
poblacion_quality_df.rename(columns={'Población en LEA': 'poblacion_lea'}, inplace=True)
poblacion_quality_df.rename(columns={'Población total': 'poblacion_total'}, inplace=True)
poblacion_quality_df.rename(columns={'Población total': 'poblacion_total'}, inplace=True)
poblacion_quality_df.rename(columns={'NOMBRE DEPARTAMENTO': 'nombre_dpto'}, inplace=True)
poblacion_quality_df.rename(columns={'NOMBRE MUNICIPIO': 'nombre_mcpio'}, inplace=True)
#Mostrar nuevos tipos y tamaño
display(poblacion_quality_df.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1159 entries, 0 to 1158
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   cod_divipola       1156 non-null   Int32   
 1   nombre_dpto        1156 non-null   category
 2   nombre_mcpio       1122 non-null   category
 3   poblacion_hogares  1156 non-null   Int32   
 4   poblacion_lea      1156 non-null   Int32   
 5   poblacion_total    1156 non-null   Int32   
dtypes: Int32(4), category(2)
memory usage: 67.9 KB


None

In [ ]:
poblacion_quality_df.to_parquet("drive/MyDrive/poblacion2018.parquet", compression="zstd")

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

# Convert Polars DataFrames to PyArrow Tables
tables = [df.to_arrow() for df in dfs]

# Concatenate PyArrow Tables
table_all = pa.concat_tables(tables)


# Write to a single Parquet file with ZSTD compression
pq.write_table(table_all, "Reportes_2023_2024.parquet", compression='zstd')
pq.write_table(table_all, "drive/MyDrive/Reportes_2023_2024.parquet", compression='zstd')

NameError: name 'dfs' is not defined

In [ ]:
Reportes_2023_2024 = pl.read_parquet("drive/MyDrive/Reportes_2023_2024.parquet")

In [ ]:
# Para borrar el campo ObjectID
#Reportes_2023_2024 = Reportes_2023_2024.drop("OBJECTID")
Reportes_2023_2024.schema

Schema([('periodo', Int64),
        ('mes', Int64),
        ('pqr_canal', String),
        ('pet_cod_depto', Int64),
        ('pet_cod_mpio', Int64),
        ('id_afec', String),
        ('afec_parentesco', String),
        ('afec_genero', String),
        ('afec_edad', Int64),
        ('afec_edadr', String),
        ('afec_educ', String),
        ('afec_regafiliacion', String),
        ('afec_getnico', String),
        ('afec_pobespecial', String),
        ('afec_cod_depto', Int64),
        ('afec_cod_mpio', Int64),
        ('ent_nombre', String),
        ('ent_tipovig_sns', String),
        ('ent_cod_sns', String),
        ('ent_alias_sn', String),
        ('ent_cod_depto', Int64),
        ('ent_cod_mpio', Int64),
        ('cod_macromot', Int64),
        ('macromotivo', String),
        ('cod_motgen', Int64),
        ('motivo_general', String),
        ('cod_motesp', Int64),
        ('motivo_especifico', String),
        ('cod_tipo_motesp', Int64),
        ('tipo_de_motivo_especifico

In [ ]:
# Supongamos que ya tienes el df_demografia cargado
# Convert PyArrow Table back to Polars DataFrame
#Reportes_2023_2024 = table_all.to_polars()

Reportes_2023_2024 = (
    Reportes_2023_2024
    .with_columns([
        pl.col("periodo").cast(pl.Int16),
        pl.col("mes").cast(pl.Int8),
        pl.col("pet_cod_depto").cast(pl.Int8),
        pl.col("pet_cod_mpio").cast(pl.Int32),
        pl.col("afec_edad").cast(pl.Int16),
        pl.col("afec_cod_depto").cast(pl.Int8),
        pl.col("afec_cod_mpio").cast(pl.Int32),
        pl.col("ent_cod_sns").cast(pl.Utf8),
        pl.col("ent_cod_depto").cast(pl.Int8),
        pl.col("ent_cod_mpio").cast(pl.Int32),
        pl.col("cod_macromot").cast(pl.Int32),
        pl.col("cod_motgen").cast(pl.Int32),
        pl.col("cod_motesp").cast(pl.Int32),
        pl.col("cod_tipo_motesp").cast(pl.Int32),
        pl.col("cod_subtipo_motesp").cast(pl.Int32),
    ])
    # Si las columnas no existen aún, se crean con nulos o valor por defecto
)
#Reportes_2023_2024=df_demografia.drop("OBJECTID")

In [ ]:
Reportes_2023_2024.schema

NameError: name 'Reportes_2023_2024' is not defined

In [ ]:
Reportes_2023_2024.write_parquet("drive/MyDrive/Reportesq_2023_2024.parquet", compression="zstd")

## Datos a procesar
 En este Punto Suponemos que los DataFrames que han sido guardados ya cuenta con un minimo de calidad de datos, homogenizacion, estandarizacion y que necesitamos solo cargarlos y procesarlos.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')
import polars as pl

reportes=pl.read_parquet("drive/MyDrive/Reportesq_2023_2024.parquet")
demografia=pl.read_parquet("drive/MyDrive/poblacion2018.parquet")

In [ ]:
reportes.shape

(2444766, 37)

In [ ]:
demografia = demografia.drop_nulls(subset=["nombre_dpto"])

In [ ]:
demografia.head(36)

cod_divipola,nombre_dpto,nombre_mcpio,poblacion_hogares,poblacion_lea,poblacion_total
i32,cat,cat,i32,i32,i32
0,"""Total Nacional""",null,43835324,329093,44164417
5,"""Antioquia""",null,5931492,43296,5974788
8,"""Atlántico""",null,2336434,5831,2342265
11,"""Bogotá, D.C.""",null,7149540,31929,7181469
13,"""Bolívar""",null,1903456,6004,1909460
…,…,…,…,…,…
95,"""Guaviare""",null,67027,6054,73081
97,"""Vaupés""",null,28963,8727,37690
99,"""Vichada""",null,71405,5237,76642


In [ ]:
# Ordenar por Departamento, agrupar por Departamento y obtener el valor máximo de Dato Numérico
demografia_max_dato = (
    demografia
    .sort("nombre_dpto")
    .group_by(["nombre_dpto"])
    .agg(pl.count().alias("Cantidad"))
    .sort("nombre_dpto")
)

display(demografia_max_dato)

In [ ]:
reportes.schema

In [ ]:
for dato in reportes['ent_cod_sns'].unique():
  print(dato)

In [ ]:
import polars as pl


# Comparaciones de Rendimiento

En el mundo del BigData el rendimiento es un factor fundamental, por esta razon al usar una u otra herramienta es fundamental saber elegir la mejor, especialmente en ambientes donde la memoria o la capacidad de procesamiento es limitada.

In [ ]:
import polars as pl
import pandas as pd
import time
import numpy as np

In [ ]:
%%capture
!pip install "polars[gpu]"

In [ ]:
%%time
reportes=pl.read_parquet("drive/MyDrive/Reportesq_2023_2024.parquet")

CPU times: user 873 ms, sys: 898 ms, total: 1.77 s
Wall time: 8.07 s


In [ ]:
%%time
reportes_pd=pd.read_parquet("drive/MyDrive/Reportesq_2023_2024.parquet")

CPU times: user 7.14 s, sys: 3.52 s, total: 10.7 s
Wall time: 26.9 s


In [ ]:
# Ejemplo de cascarón; usa tu versión final:
def ipa_polars(reportes_pl, engine="cpu"):
    # --- 1) Diccionarios de mapeo ---
    sev_map = {
        "SIMPLE": 0.00,
        "PRIORIZADO": 0.60,
        "RIESGO VITAL": 1.00,
    }

    nivel2 = {
        "Manejo de pacientes en Unidades de cuidados intensivos",
        "Trasplante de corazón", "Trasplante de hígado", "TRASPLANTE RENAL", "Trasplante renal",
        "Trasplante de pulmón", "Trasplante de médula ósea", "Manejo del trauma mayor",
        "Manejo médico quirúrgico del gran quemado"
    }
    nivel1 = {
        "Hemodiálisis", "Diálisis peritoneal", "Reemplazos articulares",
        "Quimioterapia y radioterapia para el cáncer",
        "Manejo quirúrgico para enfermedades del corazón",
        "Manejo quirúrgico para enfermedades del sistema nervioso central",
        "Implante coclear",
    }

    def costo_score_expr(col):
        return (
            pl.when(pl.col(col).is_in(nivel2)).then(pl.lit(1.0))
            .when(pl.col(col).is_in(nivel1)).then(pl.lit(0.7))
            .otherwise(pl.lit(0.0))
        )

    canal_map = {
        "Telefónico": 1.00, "Personalizado": 1.00, "Evento": 1.00,
        "Chat": 0.50, "Escrito": 0.50, "Correo": 0.50, "Web": 0.50,
        "Redes Sociales": 0.30, "Medios de comunicación": 0.30
    }

    # --- 2) Utilidad fecha ---
    def month_to_date_expr(year_col="periodo", month_col="mes"):
        return pl.datetime(year=pl.col(year_col).cast(pl.Int32),
                          month=pl.col(month_col).cast(pl.Int32),
                          day=1)

    # --- 3) LazyFrame de entrada ---
    lf = reportes.lazy()

    # (opcional) Normalización de strings para evitar espacios/diferencias de mayúsculas
    lf = lf.with_columns([
        pl.col("clasificacion_de_riesgo")
          .str.strip_chars()
          .str.replace_all(r"\s+", " ")
          .str.to_uppercase()
          .alias("clasif_norm"),
        pl.col("pqr_canal").str.strip_chars().alias("pqr_canal_norm"),
        pl.col("alto_costo").str.strip_chars().alias("alto_costo_norm"),
    ])

    # --- 4) Enriquecimiento fila a fila (ahora usando .replace) ---
    lf = (
        lf
        .with_columns([
            month_to_date_expr().alias("fecha"),
            pl.col("clasif_norm").replace_strict(sev_map, default=0.0).cast(pl.Float64).alias("sev"),
            costo_score_expr("alto_costo_norm").alias("costo"),
            pl.col("pqr_canal_norm").replace_strict(canal_map, default=0.3).cast(pl.Float64).alias("urg"),
            (pl.col("periodo")*12 + pl.col("mes")).alias("tidx"),
        ])
    )

    # --- 5) Agregación mensual ---
    grp_keys = ["ent_cod_sns", "afec_cod_depto", "fecha", "tidx"]
    mensual = (
        lf
        .group_by(grp_keys)
        .agg([
            pl.len().alias("n"),
            pl.mean("sev").alias("sev_mean"),
            pl.mean("costo").alias("costo_mean"),
            pl.mean("urg").alias("urg_mean"),
        ])
        .sort(["ent_cod_sns", "afec_cod_depto", "tidx"])
    )

    # --- 6) Rolling z-score (ventana=6) ---
    window = 6
    mensual = (
        mensual
        .with_columns([
            pl.col("n").rolling_mean(window_size=window, min_samples=3)
                .over(["ent_cod_sns", "afec_cod_depto"]).alias("n_rm"),
            pl.col("n").rolling_std(window_size=window, min_samples=3)
                .over(["ent_cod_sns", "afec_cod_depto"]).alias("n_rs"),
        ])
        .with_columns([
            pl.when(pl.col("n_rs") > 0)
              .then((pl.col("n") - pl.col("n_rm")) / pl.col("n_rs"))
              .otherwise(pl.lit(0.0))
              .alias("trend_z"),
        ])
    )

    # --- 7) Normalización de volumen por grupo ---
    mensual = (
        mensual
        .with_columns([
            pl.min("n").over(["ent_cod_sns", "afec_cod_depto"]).alias("n_min"),
            pl.max("n").over(["ent_cod_sns", "afec_cod_depto"]).alias("n_max"),
        ])
        .with_columns([
            pl.when((pl.col("n_max") - pl.col("n_min")) > 0)
              .then((pl.col("n") - pl.col("n_min")) / (pl.col("n_max") - pl.col("n_min")))
              .otherwise(pl.lit(0.0))
              .alias("vol_norm"),
        ])
    )

    # --- 8) Sigmoide 100% vectorizada (sin .apply) ---
    mensual = mensual.with_columns([
        (1 / (1 + (-pl.col("trend_z")).exp())).alias("trend_sig")
    ])

    # --- 9) Índice de Prioridad de Atención (IPA) ---
    w_sev, w_costo, w_urg, w_trend, w_vol = 0.35, 0.25, 0.15, 0.15, 0.10
    mensual = mensual.with_columns([
        (100 * (w_sev*pl.col("sev_mean")
                + w_costo*pl.col("costo_mean")
                + w_urg*pl.col("urg_mean")
                + w_trend*pl.col("trend_sig")
                + w_vol*pl.col("vol_norm"))).round(2).alias("IPA")
    ])
    return mensual.collect(engine=engine)  # DataFrame Polars final

In [ ]:
import pandas as pd
import numpy as np

# --- 1) Diccionarios / catálogos ---
sev_map = {"SIMPLE": 0.00, "PRIORIZADO": 0.60, "RIESGO VITAL": 1.00}

nivel2 = set([
    "Manejo de pacientes en Unidades de cuidados intensivos",
    "Trasplante de corazón", "Trasplante de hígado", "TRASPLANTE RENAL", "Trasplante renal",
    "Trasplante de pulmón", "Trasplante de médula ósea", "Manejo del trauma mayor",
    "Manejo médico quirúrgico del gran quemado"
])
nivel1 = set([
    "Hemodiálisis", "Diálisis peritoneal", "Reemplazos articulares",
    "Quimioterapia y radioterapia para el cáncer",
    "Manejo quirúrgico para enfermedades del corazón",
    "Manejo quirúrgico para enfermedades del sistema nervioso central",
    "Implante coclear",
])

canal_map = {
    "Telefónico": 1.00, "Personalizado": 1.00, "Evento": 1.00,
    "Chat": 0.50, "Escrito": 0.50, "Correo": 0.50, "Web": 0.50,
    "Redes Sociales": 0.30, "Medios de comunicación": 0.30
}

# --- 2) Normalización básica de strings y columnas auxiliares ---
def normaliza_col(s: pd.Series) -> pd.Series:
    # quita espacios extra y pone mayúsculas (sin tocar tildes)
    return (s.fillna("")
             .str.strip()
             .str.replace(r"\s+", " ", regex=True)
             .str.upper())

reportes_pd = reportes_pd.copy()

reportes_pd["clasif_norm"] = normaliza_col(reportes_pd["clasificacion_de_riesgo"])
reportes_pd["pqr_canal_norm"] = reportes_pd["pqr_canal"].fillna("").str.strip()
reportes_pd["alto_costo_norm"] = reportes_pd["alto_costo"].fillna("").str.strip()

# fecha y orden temporal
reportes_pd["fecha"] = pd.to_datetime(
    dict(year=reportes_pd["periodo"].astype(int), month=reportes_pd["mes"].astype(int), day=1)
)
reportes_pd["tidx"] = reportes_pd["periodo"].astype(int) * 12 + reportes_pd["mes"].astype(int)

# --- 3) Mapeos vectorizados ---
reportes_pd["sev"] = reportes_pd["clasif_norm"].map(sev_map).fillna(0.0).astype(float)

# costo por niveles (evita apply fila a fila)
alto_uc = normaliza_col(reportes_pd["alto_costo"])
reportes_pd["costo"] = np.select(
    [
        alto_uc.isin({x.upper() for x in nivel2}),
        alto_uc.isin({x.upper() for x in nivel1}),
        alto_uc.eq("NO APLICA")
    ],
    [1.0, 0.7, 0.0],
    default=0.0
).astype(float)

reportes_pd["urg"] = reportes_pd["pqr_canal_norm"].map(canal_map).fillna(0.3).astype(float)

# --- 4) Agregación mensual por grupo ---
grp = ["ent_cod_sns", "afec_cod_depto", "fecha", "tidx"]
agg = (reportes_pd
       .groupby(grp, sort=True, as_index=False)
       .agg(n=("periodo", "size"),
            sev_mean=("sev", "mean"),
            costo_mean=("costo", "mean"),
            urg_mean=("urg", "mean"))
       .sort_values(["ent_cod_sns", "afec_cod_depto", "tidx"], kind="mergesort")
)

# --- 5) Rolling z-score por grupo (ventana=6, min_periods=3) ---
window = 6
gkeys = ["ent_cod_sns", "afec_cod_depto"]
agg = agg.sort_values(gkeys + ["tidx"], kind="mergesort")

g = agg.groupby(gkeys, group_keys=False)
agg["n_rm"] = g["n"].rolling(window=window, min_periods=3).mean().reset_index(level=gkeys, drop=True)
agg["n_rs"] = g["n"].rolling(window=window, min_periods=3).std(ddof=1).reset_index(level=gkeys, drop=True)
agg["trend_z"] = np.where(agg["n_rs"].gt(0), (agg["n"] - agg["n_rm"]) / agg["n_rs"], 0.0)

# Sigmoide a [0,1]
agg["trend_sig"] = 1.0 / (1.0 + np.exp(-agg["trend_z"]))

# --- 6) Volumen normalizado (min-max por grupo histórico) ---
g = agg.groupby(gkeys)
n_min = g["n"].transform("min")
n_max = g["n"].transform("max")
den = (n_max - n_min).replace(0, np.nan)
agg["vol_norm"] = ((agg["n"] - n_min) / den).fillna(0.0)

# --- 7) IPA (mismos pesos) ---
w_sev, w_costo, w_urg, w_trend, w_vol = 0.35, 0.25, 0.15, 0.15, 0.10
agg["IPA"] = 100.0 * (
    w_sev*agg["sev_mean"] +
    w_costo*agg["costo_mean"] +
    w_urg*agg["urg_mean"] +
    w_trend*agg["trend_sig"] +
    w_vol*agg["vol_norm"]
)
agg["IPA"] = agg["IPA"].round(2)

# Resultado mensual equivalente al de Polars:
mensual_pd = agg


In [ ]:
def ipa_pandas(reportes_pd: pd.DataFrame,
               window:int=6,
               pesos=(0.35, 0.25, 0.15, 0.15, 0.10)) -> pd.DataFrame:
    sev_map = {"SIMPLE": 0.00, "PRIORIZADO": 0.60, "RIESGO VITAL": 1.00}
    # Corrected definition of nivel2 and nivel1
    nivel2 = set([
        "Manejo de pacientes en Unidades de cuidados intensivos",
        "Trasplante de corazón", "Trasplante de hígado", "TRASPLANTE RENAL", "Trasplante renal",
        "Trasplante de pulmón", "Trasplante de médula ósea", "Manejo del trauma mayor",
        "Manejo médico quirúrgico del gran quemado"
    ])
    nivel1 = set([
        "Hemodiálisis", "Diálisis peritoneal", "Reemplazos articulares",
        "Quimioterapia y radioterapia para el cáncer",
        "Manejo quirúrgico para enfermedades del corazón",
        "Manejo quirúrgico para enfermedades del sistema nervioso central",
        "Implante coclear",
    ])
    canal_map = {
        "Telefónico": 1.00, "Personalizado": 1.00, "Evento": 1.00,
        "Chat": 0.50, "Escrito": 0.50, "Correo": 0.50, "Web": 0.50,
        "Redes Sociales": 0.30, "Medios de comunicación": 0.30
    }
    def normaliza_col(s):
        return (s.fillna("").str.strip().str.replace(r"\s+", " ", regex=True).str.upper())

    df = reportes_pd.copy()
    df["clasif_norm"] = normaliza_col(df["clasificacion_de_riesgo"])
    df["pqr_canal_norm"] = df["pqr_canal"].fillna("").str.strip()
    df["alto_costo_norm"] = df["alto_costo"].fillna("").str.strip()
    df["fecha"] = pd.to_datetime(dict(year=df["periodo"].astype(int),
                                      month=df["mes"].astype(int), day=1))
    df["tidx"] = df["periodo"].astype(int) * 12 + df["mes"].astype(int)

    df["sev"] = df["clasif_norm"].map(sev_map).fillna(0.0).astype(float)
    alto_uc = normaliza_col(df["alto_costo"])
    df["costo"] = np.select(
        [alto_uc.isin({x.upper() for x in nivel2}),
         alto_uc.isin({x.upper() for x in nivel1}),
         alto_uc.eq("NO APLICA")],
        [1.0, 0.7, 0.0],
        default=0.0
    ).astype(float)
    df["urg"] = df["pqr_canal_norm"].map(canal_map).fillna(0.3).astype(float)

    grp = ["ent_cod_sns", "afec_cod_depto", "fecha", "tidx"]
    agg = (df.groupby(grp, sort=True, as_index=False)
             .agg(n=("periodo", "size"),
                  sev_mean=("sev", "mean"),
                  costo_mean=("costo", "mean"),
                  urg_mean=("urg", "mean"))
             .sort_values(["ent_cod_sns", "afec_cod_depto", "tidx"], kind="mergesort"))

    gkeys = ["ent_cod_sns", "afec_cod_depto"]
    g = agg.groupby(gkeys, group_keys=False)
    agg["n_rm"] = g["n"].rolling(window=window, min_periods=3).mean().reset_index(level=gkeys, drop=True)
    agg["n_rs"] = g["n"].rolling(window=window, min_periods=3).std(ddof=1).reset_index(level=gkeys, drop=True)
    agg["trend_z"] = np.where(agg["n_rs"].gt(0), (agg["n"] - agg["n_rm"]) / agg["n_rs"], 0.0)
    agg["trend_sig"] = 1.0 / (1.0 + np.exp(-agg["trend_z"]))

    g = agg.groupby(gkeys)
    n_min = g["n"].transform("min")
    n_max = g["n"].transform("max")
    den = (n_max - n_min).replace(0, np.nan)
    agg["vol_norm"] = ((agg["n"] - n_min) / den).fillna(0.0)

    w_sev, w_costo, w_urg, w_trend, w_vol = pesos
    agg["IPA"] = 100.0 * (
        w_sev*agg["sev_mean"] + w_costo*agg["costo_mean"] + w_urg*agg["urg_mean"]
        + w_trend*agg["trend_sig"] + w_vol*agg["vol_norm"]
    )
    agg["IPA"] = agg["IPA"].round(2)
    return agg

In [ ]:
import time

pl.Config.set_tbl_formatting("ASCII_MARKDOWN")  # ejemplo de config
# Para threads:
import os
os.environ["POLARS_MAX_THREADS"] = "2"

# Polars
t0 = time.perf_counter()
mensual_pl = ipa_polars(reportes, engine="gpu")        # 'La primera vez que se ejecute en instancia tipo CPU enviar engine="cpu",
                                                        # cuando se cambie de tipo de instancia cambiar a engine="gpu"
t1 = time.perf_counter()
time_polars = t1 - t0

# Pandas
reportes_pd = reportes.to_pandas()       # conversión justa (hazla fuera del cronómetro real)
t2 = time.perf_counter()
mensual_pd = ipa_pandas(reportes_pd)
t3 = time.perf_counter()
time_pandas = t3 - t2

print(f"Tiempo Polars: {time_polars:.3f} s")
print(f"Tiempo Pandas: {time_pandas:.3f} s")
print(f"Speedup (Pandas/Polars): {time_pandas/time_polars:.2f}×")


Tiempo Polars: 8.577 s
Tiempo Pandas: 12.506 s
Speedup (Pandas/Polars): 1.46×


In [ ]:
!pip install -q memory_profiler

In [ ]:
from memory_profiler import memory_usage

def run_polars():
    ipa_polars(reportes,"cpu")

def run_pandas():
    ipa_pandas(reportes.to_pandas())

mem_polars = max(memory_usage(run_polars, interval=0.1))
mem_pandas = max(memory_usage(run_pandas, interval=0.1))
print(f"Mem Polars (MiB): {mem_polars:.1f}")
print(f"Mem Pandas (MiB): {mem_pandas:.1f}")


Mem Polars (MiB): 5227.8
Mem Pandas (MiB): 7533.7


In [ ]:
print(f"EN gigas polars): {mem_polars/1024:.2f}g")
print(f"EN gigas pandas): {mem_pandas/1024:.2f}g")

EN gigas polars): 5.11g
EN gigas pandas): 7.36g


In [ ]:
# Ordena y compara un subconjunto de columnas clave
cols_cmp = ["ent_cod_sns","afec_cod_depto","fecha","tidx","n","sev_mean","costo_mean","urg_mean","trend_sig","vol_norm","IPA"]

# Polars -> pandas
mensual_pl_pd = mensual_pl.select(cols_cmp).to_pandas().sort_values(["ent_cod_sns","afec_cod_depto","tidx"]).reset_index(drop=True)
mensual_pd2   = mensual_pd[cols_cmp].sort_values(["ent_cod_sns","afec_cod_depto","tidx"]).reset_index(drop=True)

# diferencias promedio
diff = (mensual_pl_pd[["sev_mean","costo_mean","urg_mean","trend_sig","vol_norm","IPA"]]
        - mensual_pd2[["sev_mean","costo_mean","urg_mean","trend_sig","vol_norm","IPA"]]).abs().mean()

print(diff)


sev_mean      1.109224e-16
costo_mean    2.425147e-18
urg_mean      2.459987e-17
trend_sig     0.000000e+00
vol_norm      0.000000e+00
IPA           5.257624e-06
dtype: float64


# Streaming de datos

Se debe reiniciar el notebook, para liberar memoria e iniciar desde este punto, las librerias seran independientes,

In [ ]:
# Ejecutar al inicio del bloque práctico
!pip -q install streamz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.9/76.9 kB 1.6 MB/s eta 0:00:00


In [ ]:
import sys, asyncio, time, random, statistics
from collections import defaultdict, deque
from datetime import datetime
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

## Generador de un productor de eventos

In [ ]:
import asyncio, time, random
from collections import defaultdict
from datetime import datetime

async def productor(q: asyncio.Queue, tasa_ev_s=20):
    while True:
        await asyncio.sleep(random.expovariate(tasa_ev_s))
        ev = {"ts_proc": time.time(),
              "key": f"E{random.randint(1,30):03d}",
              "valor": random.randint(1,100)}
        await q.put(ev)


## Manejo de una ventana Tumbling

***Objetivo***. Implementar un productor de eventos y un agregador por ventanas tumbling de tamaño W=10 s, usando tiempo de procesamiento.

***Criterio de finalización***. Mostrar, cada 10 s, el conteo de eventos totales y promedio por clave.

In [ ]:
async def ventana_tumbling(q: asyncio.Queue, W=10):
    buffer = []
    t0 = time.time()
    while True:
        try:
            ev = await asyncio.wait_for(q.get(), timeout=0.1)
            buffer.append(ev)
        except asyncio.TimeoutError:
            pass
        now = time.time()
        if now - t0 >= W:
            por_key = defaultdict(list)
            for e in buffer: por_key[e["key"]].append(e["valor"])
            resumen = {k: {"n": len(v), "avg": round(sum(v)/len(v),2)} for k,v in por_key.items()}
            print(f"TUMBLING {datetime.fromtimestamp(t0).strftime('%H:%M:%S')}–{datetime.fromtimestamp(now).strftime('%H:%M:%S')} | n={len(buffer)} | keys={len(resumen)}")
            buffer.clear(); t0 = now

async def main_e1():
    q = asyncio.Queue(maxsize=5000)
    await asyncio.gather(productor(q, tasa_ev_s=25), ventana_tumbling(q, W=10))


In [ ]:
# usando await para correr el loop de procesamiento,
# detener el proceso despues de unas 10 ejecuciones y observar los resultados
await main_e1()

TUMBLING 20:12:39–20:12:49 | n=264 | keys=30
TUMBLING 20:12:49–20:12:59 | n=279 | keys=30
TUMBLING 20:12:59–20:13:09 | n=255 | keys=30


CancelledError: 

## Manejo de una ventana Sliding (Ventana Deslizante)

***Objetivo***. Añadir una ventana sliding (W=60 s, paso S=10 s) y controlar backpressure mediante cola acotada y regulación de tasa.

***Criterio de finalización***. Imprimir, cada 10 s, el número de eventos en la ventana deslizante y evidencia de regulación cuando la cola se llena.

In [ ]:
async def ventana_sliding(q: asyncio.Queue, W=60, S=10):
    from collections import deque
    win = deque()
    t_last = time.time()
    while True:
        try:
            ev = await asyncio.wait_for(q.get(), timeout=0.1)
            win.append(ev)
        except asyncio.TimeoutError:
            pass
        now = time.time()
        while win and now - win[0]["ts_proc"] > W:
            win.popleft()
        if now - t_last >= S:
            print(f"SLIDING[{W}s] n={len(win)}")
            t_last = now

async def productor_con_backpressure(q: asyncio.Queue, tasa_ev_s=60):
    # regula la tasa si la cola está llena
    while True:
        delay = random.expovariate(tasa_ev_s)
        await asyncio.sleep(delay)
        if q.full():
            await asyncio.sleep(0.05)  # alivio temporal
        await q.put({"ts_proc": time.time(), "key": f"E{random.randint(1,30):03d}", "valor": random.randint(1,100)})

async def main_e2():
    q = asyncio.Queue(maxsize=500)  # más pequeño para observar saturación
    await asyncio.gather(
        productor_con_backpressure(q, tasa_ev_s=80),  # intencionalmente alta
        ventana_sliding(q, W=60, S=10)
    )

# asyncio.run(main_e2())


In [ ]:
# usando await para correr el loop de procesamiento,
# detener el proceso despues de unas 10 ejecuciones y observar los resultados
await main_e2()

SLIDING[60s] n=734
SLIDING[60s] n=1436
SLIDING[60s] n=2184
SLIDING[60s] n=2900
SLIDING[60s] n=3674
SLIDING[60s] n=4418
SLIDING[60s] n=4431
SLIDING[60s] n=4461
SLIDING[60s] n=4526
SLIDING[60s] n=4534
SLIDING[60s] n=4531
SLIDING[60s] n=4496
SLIDING[60s] n=4456
SLIDING[60s] n=4468
SLIDING[60s] n=4376
SLIDING[60s] n=4419
SLIDING[60s] n=4369
SLIDING[60s] n=4378
SLIDING[60s] n=4404
SLIDING[60s] n=4419
SLIDING[60s] n=4464
SLIDING[60s] n=4439


CancelledError: 

## Eventos Fuera de Orden

In [ ]:
import heapq, time, random
from datetime import datetime

async def productor_out_of_order(q, tasa_ev_s=30, max_lateness_s=3.0):
    while True:
        await asyncio.sleep(random.expovariate(tasa_ev_s))
        evt = time.time() - random.uniform(0, max_lateness_s)  # ts_event atrasado
        await q.put({"ts_event": evt, "key": f"E{random.randint(1,20):03d}", "valor": random.randint(1,100)})

async def ventana_event_time_tumbling(q, W=10, L=3.0):
    # buffer como min-heap por ts_event
    buf = []
    w_start = None
    n_total = n_late_drop = n_out_of_order = 0
    while True:
        try:
            ev = await asyncio.wait_for(q.get(), timeout=0.1)
            n_total += 1
            # detectar fuera de orden
            if w_start is not None and ev["ts_event"] < w_start:
                n_out_of_order += 1
            heapq.heappush(buf, (ev["ts_event"], ev))
        except asyncio.TimeoutError:
            pass

        now = time.time()
        watermark = now - L  # todo lo anterior a watermark es "seguro"
        # inicializar ventana si corresponde
        if w_start is None and buf:
            w_start = (buf[0][0] // W) * W

        # consumir del heap mientras haya eventos con ts_event <= watermark
        # y asignarlos a su ventana; cerrar ventanas completas
        while buf and buf[0][0] <= watermark:
            ts_e, ev = heapq.heappop(buf)
            if ts_e < w_start:
                # demasiado tarde, ya cerrada su ventana
                n_late_drop += 1
                continue
            # si el evento supera el fin de la ventana actual, cerrarla y abrir siguiente
            while ts_e >= w_start + W:
                print(f"EVENT-TIME TUMBLING {datetime.fromtimestamp(w_start).strftime('%H:%M:%S')}–{datetime.fromtimestamp(w_start+W).strftime('%H:%M:%S')}"
                      f" | total={n_total} late_drop={n_late_drop} ooo={n_out_of_order}")
                w_start += W
                n_total = n_late_drop = n_out_of_order = 0

async def main_e3():
    q = asyncio.Queue(maxsize=5000)
    await asyncio.gather(productor_out_of_order(q, tasa_ev_s=40, max_lateness_s=3.0),
                         ventana_event_time_tumbling(q, W=10, L=3.0))


In [ ]:
await main_e3()

In [ ]:
# --- Ventanas TUMBLING por TIEMPO DE EVENTO con eventos atrasados y fuera de orden ---
# Colab-ready

import asyncio, time, random, math
from collections import defaultdict
from datetime import datetime

# ====== Parámetros ajustables ======
W = 8.0          # Tamaño de la ventana tumbling en segundos (p. ej., 10 s)
L = 1.0           # Tolerancia de atraso (lateness) en segundos para el watermark
RATE = 70         # Tasa media de eventos por segundo (llegadas ~Poisson)
P_LATE = 0.35     # Probabilidad de que un evento sea realmente "tarde" (más que L)
LATE_MIN = 2.0    # Límite inferior del atraso cuando un evento llega tarde (> L para forzar late_drop)
LATE_MAX = 6.0    # Límite superior del atraso cuando un evento llega tarde
RUNTIME = 30      # Duración de la demostración en segundos
# ===================================

def win_start(ts, W):
    """Devuelve el inicio de la ventana (múltiplo de W) para un timestamp dado."""
    return math.floor(ts / W) * W

async def producer(q: asyncio.Queue):
    """
    Productor de eventos con llegadas ~exponenciales (Poisson).
    - Una fracción P_LATE de eventos se generan con ts_event en el pasado por más de L,
      para que caigan en ventanas ya cerradas (late_drop).
    - El resto tiene pequeñas perturbaciones (0–0.5 s) para ocasionar "out-of-order" ocasional.
    """
    start = time.time()
    while time.time() - start < RUNTIME:
        # Tiempo entre llegadas ~Exponential(λ=RATE)
        await asyncio.sleep(random.expovariate(RATE))
        now = time.time()

        if random.random() < P_LATE:
            # Evento "realmente tarde": su tiempo de evento es varios segundos antes de 'now'
            # (más de L) para asegurar que muchas veces su ventana ya esté cerrada.
            ev_time = now - random.uniform(LATE_MIN, LATE_MAX)
        else:
            # Evento "en orden" con pequeña variación; puede quedar fuera de orden si llega
            # un evento con ts_event levemente menor después.
            ev_time = now - random.uniform(0.0, 0.5)

        ev = {
            "ts_event": ev_time,                # Momento real del evento (tiempo de evento)
            "ts_ingest": now,                   # Momento de observación/procesamiento
            "key": f"E{random.randint(1,20):03d}",
            "value": random.randint(1,100),
        }
        await q.put(ev)

async def event_time_tumbling(q: asyncio.Queue):
    """
    Mantenedor de ventanas TUMBLING por TIEMPO DE EVENTO.
    - Watermark = max_event_ts - L
    - Se cierran/emitén ventanas cuyo fin (ws+W) sea <= watermark.
    - Un evento que cae en una ventana ya emitida se cuenta como late_drop.
    - Se contabiliza out-of-order si el ts_event decrece respecto del último visto.
    """

    # Estado por ventana
    buckets = defaultdict(int)   # ws -> conteo de eventos dentro de la ventana abierta
    late_drop = defaultdict(int) # ws -> conteo de eventos tardíos descartados para esa ventana

    emitted_until = None         # Marca del último "fin de ventana" ya emitido (float segundos)

    max_event_ts = float("-inf")     # Máximo ts_event visto (para calcular watermark)
    last_seen_event_ts = float("-inf")  # Último ts_event observado (para detectar out-of-order)
    ooo_count = 0                   # Conteo global de eventos fuera de orden

    start = time.time()
    # Se extiende el lazo un poco más allá de RUNTIME para drenar eventos tardíos
    while time.time() - start < RUNTIME + 2*LATE_MAX:

        # 1) Intentar leer un evento (no bloquear indefinidamente)
        try:
            ev = await asyncio.wait_for(q.get(), timeout=0.1)
            et = ev["ts_event"]

            # 1.a) Detección de "out-of-order": si el ts_event bajó vs último visto
            if et < last_seen_event_ts:
                ooo_count += 1
            # Actualizar "último visto" con el máximo (no perder el mayor)
            last_seen_event_ts = max(last_seen_event_ts, et)

            # 1.b) Actualizar el máximo ts_event para el watermark
            if et > max_event_ts:
                max_event_ts = et

            # 1.c) Determinar la ventana del evento por TIEMPO DE EVENTO
            ws = win_start(et, W)  # inicio de la ventana
            we = ws + W            # fin de la ventana

            # 1.d) Si la ventana ya fue emitida (cerrada), el evento es "late_drop"
            if emitted_until is not None and we <= emitted_until:
                late_drop[ws] += 1
            else:
                buckets[ws] += 1

        except asyncio.TimeoutError:
            pass

        # 2) Cierre de ventanas por watermark (evento más reciente visto - L)
        if max_event_ts != float("-inf"):
            watermark = max_event_ts - L

            # Ventanas cerrables: aquellas cuyo fin <= watermark
            closable = sorted([ws for ws in buckets.keys() if (ws + W) <= watermark])

            # Inicializar 'emitted_until' si aún no se ha emitido ninguna ventana
            if emitted_until is None and closable:
                emitted_until = 0.0  # cualquier tiempo anterior al primer cierre real

            for ws in closable:
                we = ws + W
                # Emitir resumen de la ventana
                n_inside = buckets.pop(ws, 0)
                n_late = late_drop.pop(ws, 0)
                print(
                    f"EVENT-TIME TUMBLING "
                    f"{datetime.fromtimestamp(ws).strftime('%H:%M:%S')}–{datetime.fromtimestamp(we).strftime('%H:%M:%S')} "
                    f"| inside={n_inside} late_drop={n_late} ooo_total={ooo_count}"
                )
                # Avanzar la marca de "hasta dónde se ha emitido"
                emitted_until = max(emitted_until, we)

        # 3) Pequeña espera para evitar un bucle muy ajustado
        await asyncio.sleep(0.001)

    # 4) Drenaje final: cerrar cualquier ventana restante cuyo fin <= último watermark
    if max_event_ts != float("-inf"):
        watermark = max_event_ts - L
        closable = sorted([ws for ws in buckets.keys() if (ws + W) <= watermark])
        for ws in closable:
            we = ws + W
            n_inside = buckets.pop(ws, 0)
            n_late = late_drop.pop(ws, 0)
            print(
                f"[DRAIN] EVENT-TIME TUMBLING "
                f"{datetime.fromtimestamp(ws).strftime('%H:%M:%S')}–{datetime.fromtimestamp(we).strftime('%H:%M:%S')} "
                f"| inside={n_inside} late_drop={n_late} ooo_total={ooo_count}"
            )

async def main():
    """Orquestación: lanza el productor y el agregador de ventanas."""
    q = asyncio.Queue(maxsize=5000)  # Cola con capacidad amplia; ajustar para probar backpressure
    await asyncio.gather(
        producer(q),
        event_time_tumbling(q),
    )



In [ ]:
await main()

EVENT-TIME TUMBLING 20:26:32–20:26:40 | inside=44 late_drop=0 ooo_total=63
EVENT-TIME TUMBLING 20:26:40–20:26:48 | inside=465 late_drop=0 ooo_total=498
EVENT-TIME TUMBLING 20:26:48–20:26:56 | inside=479 late_drop=0 ooo_total=966
EVENT-TIME TUMBLING 20:26:56–20:27:04 | inside=477 late_drop=0 ooo_total=1389
